# Energy-based OOD Detection — Liu et al. 2020

**Before running:** Runtime → Change runtime type → **T4 GPU**

| Step | Script | Est. time (T4) |
|------|--------|----------------|
| 1 | Pre-train WRN-40-2 | ~90 min |
| 2 | Fine-tune with energy margin | ~12 min |
| 3 | Evaluate | ~5 min |

Checkpoints are saved to Google Drive so you can resume if the session disconnects.

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
# ── Cell 2: Mount Google Drive (for saving checkpoints) ──────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/csc8851_energy_ood'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive folder:', DRIVE_DIR)

In [ ]:
# ── Cell 3: Clone repo and set up ────────────────────────────────────────
import os

REPO = 'csc8851-vqvae-ood'
if not os.path.exists(REPO):
    !git clone https://github.com/harshitjain25/csc8851-vqvae-ood.git
else:
    !cd {REPO} && git pull

%cd {REPO}
!pip install -q torch torchvision scikit-learn numpy
print('Setup complete')

In [ ]:
# ── Cell 4: Restore checkpoints from Drive (if resuming) ─────────────────
import shutil, os

os.makedirs('checkpoints', exist_ok=True)
for fname in ['wrn40_cifar10_best.pth', 'wrn40_cifar10_last.pth', 'wrn40_energy_ft.pth']:
    src = f'{DRIVE_DIR}/{fname}'
    dst = f'checkpoints/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'Restored: {fname}')

print('Checkpoint dir:', os.listdir('checkpoints'))

In [ ]:
# ── Cell 5: Download 300K Random Images (~2.8 GB, ~5-10 min) ─────────────
# This is the standard substitute for 80M Tiny Images (taken down 2020).
# Skip this cell if data/300K_random_images.npy already exists.
import os

os.makedirs('data', exist_ok=True)
NPY_PATH = 'data/300K_random_images.npy'

if os.path.exists(NPY_PATH):
    print('Already downloaded:', NPY_PATH)
else:
    print('Downloading 300K Random Images (~2.8 GB)...')
    !wget -q --show-progress -O {NPY_PATH} \
        https://people.eecs.berkeley.edu/~hendrycks/300K_random_images.npy
    print('Done:', NPY_PATH)

## Step 1 — Pre-train WideResNet-40-2 on CIFAR-10
100 epochs · SGD · cosine LR · target ~94% accuracy · ~90 min on T4

The script saves a checkpoint after every epoch (`checkpoints/wrn40_cifar10_last.pth`)
and the best checkpoint separately. It **auto-resumes** from the last checkpoint if the
session disconnects — just re-run Cell 4 (restore from Drive) then this cell.

In [ ]:
# ── Cell 6: Pre-train WRN-40-2 ───────────────────────────────────────────
!python train_wrn.py

In [ ]:
# ── Cell 7: Back up pre-training checkpoints to Drive ────────────────────
import shutil
for fname in ['wrn40_cifar10_best.pth', 'wrn40_cifar10_last.pth']:
    src = f'checkpoints/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_DIR}/{fname}')
        print(f'Backed up: {fname}')

## Step 2 — Fine-tune with energy-bounded learning
10 epochs · SGD · cosine LR · λ=0.1 · m_in=−23 · m_out=−5 · ~12 min on T4

This adds the energy margin loss on top of cross-entropy, using 300K Random Images
as auxiliary OOD training data (substitute for 80M Tiny Images).

In [ ]:
# ── Cell 8: Energy fine-tuning ───────────────────────────────────────────
!python train_energy_finetune.py

In [ ]:
# ── Cell 9: Back up fine-tuned checkpoint to Drive ───────────────────────
import shutil
src = 'checkpoints/wrn40_energy_ft.pth'
if os.path.exists(src):
    shutil.copy(src, f'{DRIVE_DIR}/wrn40_energy_ft.pth')
    print('Backed up: wrn40_energy_ft.pth')

## Step 3 — Evaluate
Evaluates both models on SVHN, Textures/DTD, and CIFAR-100.
Results are printed as a table and saved to `outputs/energy_results.txt`.

In [ ]:
# ── Cell 10: Evaluate ────────────────────────────────────────────────────
!python evaluate_energy.py

In [ ]:
# ── Cell 11: Back up results to Drive ────────────────────────────────────
import shutil
os.makedirs('outputs', exist_ok=True)
src = 'outputs/energy_results.txt'
if os.path.exists(src):
    shutil.copy(src, f'{DRIVE_DIR}/energy_results.txt')
    print('Results saved to Drive')

# Print results
with open(src) as f:
    print(f.read())